# Training a real network, tracked in Weights & Biases

In module 02 you trained toy models on made-up data. Here you train real image classifiers on Fashion-MNIST: 70,000 small grayscale photos of clothing in 10 classes. The dataset is small enough to train in minutes on your Mac and hard enough that architecture and training choices visibly matter. Along the way you pick up the habits that separate real training from toy loops, and every run gets logged to Weights & Biases (W&B) so you can compare runs side by side.

## In this notebook

1. The data: load Fashion-MNIST and turn it into tensors.
2. Three splits: train, validation, and test, and why the test set gets touched once.
3. Cross-entropy loss and the sanity check every classifier should pass at step 0.
4. Initialization and healthy gradients: why deep networks go silent or blow up, how Kaiming and Xavier init prevent it, the diagnostics that catch it, and gradient clipping.
5. A reusable training loop with validation, W&B logging, and gradient-norm tracking.
6. MLPs vs. convolutional networks.
7. Overfitting, and the two standard fixes (dropout and weight decay).
8. Learning-rate schedules with warmup.
9. Saving and reloading checkpoints, and the one-time test.
10. A hyperparameter sweep (optional).

## Before you start

- Finish module 02 first (tensors, autograd, the `nn` training loop). Its sections 9 to 11 (gradient descent, noisy gradients, Adam) come up here, and module 03's section 8 (backprop with matrices) helps with the gradient math in section 4.
- Log in to W&B once with `uv run wandb login`, using your personal account. Every run in this notebook logs to the project `ml-lab-fashion` and prints a link. Keep the project page open in a browser tab and compare runs there as you go.
- The first run downloads Fashion-MNIST from the Hugging Face Hub (well under 100 MB, cached after that).
- Time: roughly 15 to 25 minutes of training for the whole notebook on your Mac. The optional sweep at the end adds about 10 more.
- Saved models go in a `checkpoints/` folder inside this module, which `.gitignore` already excludes, so weights never end up in git.
- Kernel: the `.venv` in `ML_Lab`.

In [ ]:
# One-time step: adds pillow (the image library datasets uses to decode the photos) to ML_Lab's
# pyproject.toml and uv.lock. Safe to rerun. uv finds the ML_Lab project from this folder.
!uv add pillow

In [ ]:
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import wandb
from datasets import load_dataset

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed(0)
PROJECT = "ml-lab-fashion"

## 1. The data

In [ ]:
ds = load_dataset("zalando-datasets/fashion_mnist")
print(ds)
CLASSES = ds["train"].features["label"].names
print(CLASSES)

The images come back as PIL images. Convert everything to tensors once, scaled to the range 0 to 1. The whole training set is about 190 MB as float32, so it fits in memory easily.

In [ ]:
def to_tensors(split):
    images = np.stack([np.array(img, dtype=np.uint8) for img in split["image"]])
    X = torch.from_numpy(images).float() / 255.0
    y = torch.tensor(list(split["label"]))
    return X, y

start = time.time()
X_full, y_full = to_tensors(ds["train"])
X_test, y_test = to_tensors(ds["test"])
print(f"train {tuple(X_full.shape)}  test {tuple(X_test.shape)}  ({time.time() - start:.1f}s)")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(X_full[i], cmap="gray")
    ax.set_title(CLASSES[y_full[i]], fontsize=9)
    ax.axis("off")
plt.show()

## 2. Three splits

- Training set: the model learns from it.
- Validation set: you look at it to make decisions (architecture, learning rate, when to stop). Carved out of the original training data.
- Test set: touched once, at the very end, to get an honest estimate of real-world accuracy. If you choose models by test score, the test score stops being honest, because you've indirectly fit it.

Normalization uses statistics from the training set only. Using validation or test statistics would leak information about them into training.

In [ ]:
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(X_full), generator=g)
n_val = len(X_full) // 6                     # 10,000 of the 60,000
val_idx, train_idx = perm[:n_val], perm[n_val:]

X_train, y_train = X_full[train_idx], y_full[train_idx]
X_val, y_val = X_full[val_idx], y_full[val_idx]

mean, std = X_train.mean(), X_train.std()
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print(f"train {len(X_train)}  val {len(X_val)}  test {len(X_test)}")
print("examples per class in train:", torch.bincount(y_train).tolist())

A `DataLoader` serves the data in shuffled mini-batches. Each optimizer step uses one batch, and one pass over all batches is an epoch. Because each step sees only 128 images, its gradient is a noisy estimate of the gradient over the full training set: right on average, off by a random amount on any single step (module 02, section 10).

In [ ]:
def make_loader(X, y, batch_size, shuffle):
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, 128, shuffle=True)
val_loader = make_loader(X_val, y_val, 512, shuffle=False)
test_loader = make_loader(X_test, y_test, 512, shuffle=False)

xb, yb = next(iter(train_loader))
print("batch:", tuple(xb.shape), tuple(yb.shape), "| steps per epoch:", len(train_loader))

## 3. Cross-entropy, and the step-0 sanity check

A classifier outputs one logit per class. Softmax turns logits into probabilities, and cross-entropy is the negative log of the probability assigned to the correct class. Confident and right gives a loss near 0. Confident and wrong gives a big loss.

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0]])
target = torch.tensor([0])

probs = torch.softmax(logits, dim=-1)
print("probabilities:", probs)
print("by hand:      ", -torch.log(probs[0, target]).item())
print("F.cross_entropy:", F.cross_entropy(logits, target).item())

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=256, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                       # (N, 28, 28) -> (N, 784)
            nn.Linear(784, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 10),
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
with torch.no_grad():
    loss0 = F.cross_entropy(model(xb.to(device)), yb.to(device))
print(f"loss at step 0: {loss0.item():.3f}   expected for 10 classes: -ln(1/10) = {math.log(10):.3f}")

A freshly initialized network should be roughly uniform across the classes, so its loss should be close to ln(10) ≈ 2.30. If your step-0 loss is far off (say 15), something is wrong with initialization or the scale of the inputs, and you've found it before wasting a training run. Check this on every new model.

## 4. Initialization and healthy gradients

The step-0 check catches gross problems, like a loss of 15 instead of 2.30. Deeper networks have a subtler one. Every layer multiplies its input by a weight matrix, so if the weights are a little too large, the signal grows at every layer on the way forward, and if they're a little too small, it shrinks. Over 10 or 50 layers, "a little" compounds. Backprop multiplies by the same weights (transposed) on the way back, so gradients do the same thing as they travel from the loss to the first layer: they shrink toward zero (vanishing gradients) or blow up (exploding gradients). A network in either state can pass the step-0 check and still be untrainable.

### Why the scale compounds

Take one neuron in a layer with $n$ inputs. Its pre-activation, the number that goes into the activation function, is

$$z = \sum_{k=1}^{n} w_k \, x_k$$

where $x_1, \dots, x_n$ are the inputs coming from the previous layer and $w_1, \dots, w_n$ are this neuron's weights. Assume the weights and inputs are independent random numbers with mean zero. The variance $\mathrm{Var}(\cdot)$ measures spread: it's the average squared distance from the mean, and the standard deviation is its square root.

Two facts about variance do all the work. The variance of a sum of independent terms is the sum of their variances. And for independent zero-mean $w$ and $x$, $\mathrm{Var}(wx) = \mathrm{Var}(w)\,\mathrm{Var}(x)$. The sum has $n$ identical terms, so

$$\mathrm{Var}(z) = n \, \mathrm{Var}(w) \, \mathrm{Var}(x)$$

In words: each layer multiplies the variance of the signal by $n\,\mathrm{Var}(w)$. If that factor is 1, the scale stays put, which requires

$$\mathrm{Var}(w) = \frac{1}{n}, \quad \text{that is, a standard deviation of } \frac{1}{\sqrt{n}}$$

Stack $L$ layers and the factors multiply:

$$\mathrm{Var}(z_L) = \big(n \, \mathrm{Var}(w)\big)^{L} \, \mathrm{Var}(x)$$

That's exponential in depth. With $n = 256$ and weights drawn with standard deviation 1, the factor is 256 per layer. With standard deviation 0.01 it's $256 \times 0.0001 \approx 0.026$. Ten layers turn those into roughly $10^{24}$ and $10^{-16}$. No learning rate rescues either network.

### The activation function changes the answer

ReLU sets every negative input to zero. If $z$ is symmetric around zero, that throws away half of it. The careful version of the calculation uses the average of $x^2$ instead of $\mathrm{Var}(x)$ (ReLU outputs are never negative, so their mean isn't zero), and the average of $\mathrm{ReLU}(z)^2$ is exactly half of $\mathrm{Var}(z)$. So a ReLU layer multiplies the variance by $n\,\mathrm{Var}(w)/2$, and you make up for the lost half by doubling the weight variance:

$$\text{Kaiming (He) init:} \quad \mathrm{Var}(w) = \frac{2}{n_{\text{in}}}$$

Here $n_{\text{in}}$ is the layer's number of inputs, called its fan-in. In PyTorch: `nn.init.kaiming_normal_(w, nonlinearity="relu")`, which draws from a normal distribution with standard deviation $\sqrt{2}/\sqrt{n_{\text{in}}}$. The $\sqrt{2}$ is what PyTorch calls the gain for ReLU.

The backward pass has its own version of the problem. Backprop sends the gradient through $W^\top$, the transposed weight matrix, and each input of a layer collects gradient from all $n_{\text{out}}$ outputs. Keeping the gradient's scale steady therefore wants $\mathrm{Var}(w) = 1/n_{\text{out}}$. Glorot and Bengio (2010) balanced the two:

$$\text{Xavier (Glorot) init:} \quad \mathrm{Var}(w) = \frac{2}{n_{\text{in}} + n_{\text{out}}}$$

where $n_{\text{out}}$ is the layer's number of outputs (its fan-out). For a square layer this is just $1/n$. It was derived for activations that act like the identity near zero, such as tanh. PyTorch's `nn.init.xavier_normal_(w, gain=...)` multiplies the standard deviation by a gain, and `nn.init.calculate_gain("tanh")` returns 5/3 to make up for tanh squashing its inputs, the same job the $\sqrt{2}$ does for ReLU.

Two ways an individual unit stops learning:

- A saturated tanh unit. $\tanh$ flattens out near $\pm 1$, and its derivative is $1 - \tanh^2(z)$, which is nearly 0 when $\tanh(z)$ is near $\pm 1$. Backprop multiplies the incoming gradient by that derivative, so a saturated unit passes almost no gradient back and its incoming weights barely change.
- A dead ReLU. ReLU's derivative is 1 for positive inputs and 0 for negative ones. A unit whose input is negative for every example always outputs 0 and always gets zero gradient, so its weights never change again.

What about PyTorch's default? `nn.Linear` draws its weights uniformly from $\left(-\frac{1}{\sqrt{n_{\text{in}}}}, \frac{1}{\sqrt{n_{\text{in}}}}\right)$. A uniform distribution on $(-a, a)$ has variance $a^2/3$, so the standard deviation is $1/\sqrt{3\,n_{\text{in}}}$ and the per-layer factor is $n_{\text{in}}\mathrm{Var}(w) = 1/3$, or $1/6$ behind a ReLU. That's fine for shallow networks like the MLP and CNN in this notebook, and for networks with normalization layers, but too small for a deep plain ReLU stack.

### Experiment: a 10-layer network at four initializations

The next cell builds a deliberately bad case: a plain MLP with 10 hidden layers of width 256 and no normalization, run on 512 real training images from section 2. `probe` does one forward pass, records each hidden layer's activations, runs one backward pass of the cross-entropy loss, and records three things per hidden layer:

- the standard deviation of its activations (forward signal),
- the norm of the gradient arriving at its output, $\partial L / \partial h$ (backward signal),
- the norm of the gradient for its weight matrix (what the optimizer actually uses).

These experiments are small, so they run on the CPU. The last line checks the default-init formula from above against PyTorch.

In [ ]:
DEPTH, WIDTH = 10, 256
X_probe, y_probe = X_train[:512], y_train[:512]   # real, normalized training images


def deep_mlp(act):
    # 784 -> 10 hidden layers of 256 -> 10, with no normalization and no residual connections
    layers, n_in = [nn.Flatten()], 784
    for _ in range(DEPTH):
        layers += [nn.Linear(n_in, WIDTH), act()]
        n_in = WIDTH
    layers.append(nn.Linear(WIDTH, 10))
    return nn.Sequential(*layers)


def init_weights(model, scheme):
    for layer in model:
        if not isinstance(layer, nn.Linear) or scheme == "pytorch default":
            continue   # "pytorch default" keeps what nn.Linear did when it was created
        if scheme == "std 1.0":
            nn.init.normal_(layer.weight, 0.0, 1.0)
        elif scheme == "std 0.01":
            nn.init.normal_(layer.weight, 0.0, 0.01)
        elif scheme == "kaiming":
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        elif scheme == "xavier":
            nn.init.xavier_normal_(layer.weight, gain=nn.init.calculate_gain("tanh"))
        nn.init.zeros_(layer.bias)
    return model


def probe(model, X, y):
    model.zero_grad()
    acts, h = [], X
    for layer in model:
        h = layer(h)
        if isinstance(layer, (nn.ReLU, nn.Tanh)):
            h.retain_grad()   # autograd normally discards gradients of intermediate tensors
            acts.append(h)
    loss = F.cross_entropy(h, y)
    loss.backward()
    hidden = [layer for layer in model if isinstance(layer, nn.Linear)][:-1]
    return {
        "loss": loss.item(),
        "acts": [a.detach() for a in acts],
        "act_std": [a.std().item() for a in acts],
        "out_grad": [a.grad.norm().item() for a in acts],
        "weight_grad": [layer.weight.grad.norm().item() for layer in hidden],
    }


print(f"default nn.Linear(256, 256) weight std: {nn.Linear(256, 256).weight.std().item():.4f}   "
      f"formula 1/sqrt(3 * 256) = {1 / math.sqrt(3 * 256):.4f}")

Predict before you run the next cell. For each of the four ReLU networks below, work out the per-layer variance factor $n\,\mathrm{Var}(w)/2$ with $n = 256$, and its square root, the factor for the standard deviation. Will the activation std grow, shrink, or stay flat with depth? Which network will pass section 3's step-0 check, with a loss near 2.30?

In [ ]:
relu_results = {}
for scheme in ["std 1.0", "std 0.01", "pytorch default", "kaiming"]:
    set_seed(0)
    relu_results[scheme] = probe(init_weights(deep_mlp(nn.ReLU), scheme), X_probe, y_probe)

depths = range(1, DEPTH + 1)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for scheme, r in relu_results.items():
    axes[0].semilogy(depths, r["act_std"], "o-", label=scheme)
    axes[1].semilogy(depths, r["out_grad"], "o-", label=scheme)
    axes[2].semilogy(depths, r["weight_grad"], "o-", label=scheme)
axes[0].set_title("forward: activation std")
axes[1].set_title("backward: gradient norm at the layer's output")
axes[2].set_title("weight gradient norm")
for ax in axes:
    ax.set_xlabel("hidden layer")
axes[0].legend(fontsize=8)   # same colors in all three panels
plt.show()

for scheme, r in relu_results.items():
    dead = (r["acts"][-1] <= 0).all(dim=0).float().mean().item()   # zero for every image in the batch
    print(f"{scheme:16} step-0 loss {r['loss']:9.3g} | weight grad norms {min(r['weight_grad']):.1e} to "
          f"{max(r['weight_grad']):.1e} | dead ReLUs in layer {DEPTH}: {dead:.0%}")

What to notice:

- Standard deviation 1.0: the factor is $256 \times 1 / 2 = 128$ in variance, so about $\sqrt{128} \approx 11$ in standard deviation, and that's what the left plot shows: each layer is about 11 times larger than the one before, passing $10^{10}$ by layer 10. The step-0 loss is around $10^{12}$, and the weight gradients are around $10^{11}$. One optimizer step at a normal learning rate would wreck the network.
- Standard deviation 0.01: the mirror image. The factor is $256 \times 0.0001 / 2 = 0.0128$, about 0.11 in standard deviation, so the signal fades to around $10^{-10}$ by layer 10 and the backward gradient fades the same way on its way to layer 1. The step-0 loss is a perfect 2.30, because tiny logits give a uniform softmax. Section 3's check passes, yet every weight gradient is around $10^{-9}$ and nothing can learn. This is why the step-0 check isn't enough on its own.
- PyTorch default: the activations shrink by about $\sqrt{1/6} \approx 0.4$ per layer at first, then level off around 0.02, where the small random biases take over. The gradient reaching layer 1 is thousands of times smaller than at layer 10, and more than half of the last layer's units are dead for this batch.
- Kaiming: activation std stays roughly level and the backward gradient is flat across depth. That's what healthy looks like. Its step-0 loss is a bit above 2.30 because the output layer also got the ReLU gain, which spreads the logits more than needed. A common fix is to scale the last layer's weights down (Karpathy does this in his makemore videos).

Why are the weight gradients in the right-hand plot nearly flat across depth, even for std 1.0 and std 0.01, whose backward gradients change by 9 or 10 orders of magnitude? The gradient for a weight matrix is $dW = X^\top dZ$ (module 03, section 8): the activations coming into the layer, $X$, times the gradient arriving at the layer's pre-activation, $dZ$. Going deeper, $X$ grows by about the same factor per layer that $dZ$ shrinks (or the reverse, for std 0.01), so their product stays at roughly the same size. What's broken is the level: compare the printed ranges.

One more detail: std 1.0, std 0.01, and Kaiming have the same dead fraction. With zero biases, multiplying all weights by a constant scales every activation without changing its sign. The init's job is the scale, not the pattern.

### Saturation, seen directly

Now the same experiment with tanh. Predict: with standard deviation 1.0, a tanh unit's pre-activation has a standard deviation of about $\sqrt{256} = 16$. Where does $\tanh$ of a number that big land? The plot shows histograms of every other hidden layer's activations, for the bad init and for Xavier with PyTorch's tanh gain. A value beyond $\pm 0.97$ counts as saturated: past that point the derivative $1 - \tanh^2$ is below 0.06. The printout also gives each network's backward gradient norm at the last and the first hidden layer.

In [ ]:
tanh_results = {}
for scheme in ["std 1.0", "std 0.01", "pytorch default", "xavier"]:
    set_seed(0)
    tanh_results[scheme] = probe(init_weights(deep_mlp(nn.Tanh), scheme), X_probe, y_probe)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, scheme in zip(axes, ["std 1.0", "xavier"]):
    for i in range(1, DEPTH, 2):   # hidden layers 2, 4, 6, 8, 10
        counts, edges = torch.histogram(tanh_results[scheme]["acts"][i].flatten(), bins=50, range=(-1.0, 1.0))
        ax.plot((edges[:-1] + edges[1:]) / 2, counts / counts.sum(), label=f"layer {i + 1}")
    ax.set_title(f"tanh activations, {scheme} init")
    ax.set_xlabel("activation value")
    ax.legend(fontsize=8)
axes[0].set_ylabel("fraction of values")
plt.show()

for scheme, r in tanh_results.items():
    a = r["acts"][-1]
    saturated = (a.abs() > 0.97).float().mean().item()
    print(f"{scheme:16} layer {DEPTH}: activation std {a.std().item():.1e} | saturated {saturated:4.0%} | "
          f"average 1 - tanh^2 {(1 - a**2).mean().item():.2f} | gradient norm at layer {DEPTH} "
          f"{r['out_grad'][-1]:.1e}, at layer 1 {r['out_grad'][0]:.1e}")

What to notice:

- With standard deviation 1.0, the histograms pile up at $-1$ and $+1$ in every layer. Most of the units are saturated, and the average local derivative $1 - \tanh^2$ is around 0.05, so on average a unit passes on about 5% of the gradient it receives. Yet the gradient as a whole doesn't vanish. It grows on the way back, from about 1 at layer 10 to over $10^4$ at layer 1, because the few unsaturated units pass gradient through weights that are far too large. This network has both problems at once: the saturated units' incoming weights hardly get any gradient, and the gradient norm still explodes toward the input.
- Standard deviation 0.01 and the PyTorch default have no saturation at all, but their activations shrink with depth (the default levels off around 0.04, held up by its random biases), and their gradients shrink on the way back to layer 1: the vanishing problem from the ReLU experiment, now in a tanh network.
- Xavier with the tanh gain keeps the activations spread across the range at every depth, with only a few percent saturated, and its gradient norm changes by only about a factor of 2 between layer 10 and layer 1.

### Diagnostics you can run on any model

These four checks catch most initialization and learning-rate problems. They're cheap, so you can compute them every few hundred steps in a real run.

1. Activation standard deviation per layer. It should stay roughly level with depth, not race toward 0 or grow layer by layer. A forward hook (`module.register_forward_hook`) lets you capture activations without rewriting the model.
2. The fraction of saturated tanh units ($|a| > 0.97$) or dead ReLUs (units that output 0 for every example in a batch). A few percent is normal. Half a layer is a problem.
3. Gradient norm per layer: after `loss.backward()`, `p.grad.norm()` for each entry in `model.named_parameters()`. Look for layers whose gradients are orders of magnitude smaller or larger than the rest.
4. The update-to-weight ratio, which measures how big each step is compared with the weights it changes:

$$\text{ratio} = \frac{\lVert \Delta W \rVert}{\lVert W \rVert}$$

$W$ is a weight matrix, $\Delta W$ is the change one optimizer step made to it, and $\lVert \cdot \rVert$ is the norm (the square root of the sum of the squared entries, the matrix's overall size). For plain SGD, $\Delta W = -\eta \, \nabla_W L$, where $\eta$ is the learning rate and $\nabla_W L$ is the gradient, so the ratio is $\eta \lVert \nabla_W L \rVert / \lVert W \rVert$: learning rate times gradient norm over weight norm. With Adam the step isn't simply $\eta$ times the gradient, so it's easier to measure $\Delta W$ directly. A common rule of thumb is about $10^{-3}$ per step. Much smaller and the layer is barely learning. Much larger and each step rewrites a noticeable fraction of the weights.

The next cell trains the healthy Kaiming network for 200 steps with AdamW at three learning rates and measures the ratio for every layer, averaged over the last 20 steps.

In [ ]:
def update_ratios(model, opt, X, y):
    # One optimizer step; returns ||delta W|| / ||W|| for each Linear layer's weight matrix.
    linears = [layer for layer in model if isinstance(layer, nn.Linear)]
    before = [layer.weight.detach().clone() for layer in linears]
    loss = F.cross_entropy(model(X), y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    return loss.item(), [((layer.weight.detach() - w0).norm() / w0.norm()).item() for layer, w0 in zip(linears, before)]


for lr in [1e-4, 1e-3, 1e-2]:
    set_seed(0)
    model = init_weights(deep_mlp(nn.ReLU), "kaiming")
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(0)
    losses, ratios = [], []
    for step in range(200):
        idx = torch.randint(0, len(X_train), (128,), generator=g)
        loss, r = update_ratios(model, opt, X_train[idx], y_train[idx])
        losses.append(loss)
        ratios.append(r)
    recent = np.mean(ratios[-20:], axis=0)   # one number per layer
    print(f"lr {lr:.0e}: update/weight ratio median {np.median(recent):.1e} (layers range {recent.min():.1e} "
          f"to {recent.max():.1e}) | loss over the last 20 steps {np.mean(losses[-20:]):.3f}")

What to notice: the ratio tracks the learning rate. At 1e-3, the rate this notebook uses with AdamW, it sits close to the $10^{-3}$ rule of thumb. At 1e-4 it's about ten times smaller and the weights are barely moving, so after the same 200 steps the loss is behind. At 1e-2 the steps are several times larger than the rule of thumb, and the loss ends up worse, not better. In a real run you'd log these ratios per layer and watch for one layer drifting far from the others.

### Exploding gradients and gradient clipping

Even with a good init, an unlucky batch or a learning rate on the edge can produce one huge gradient. A huge gradient means a huge step, the step can land somewhere with an even larger loss and gradient, and a few steps later the loss is `nan`. Gradient clipping puts a cap on the step. Treat all of the model's gradients as one long vector $g$ and compute its length $\lVert g \rVert$, the global gradient norm. If the length is above a threshold $c$, rescale:

$$g \leftarrow g \cdot \frac{c}{\lVert g \rVert} \qquad \text{if } \lVert g \rVert > c$$

Multiplying by the positive number $c / \lVert g \rVert$ keeps the direction and sets the length to exactly $c$. Gradients shorter than $c$ pass through untouched. In PyTorch this is `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=c)`, called between `loss.backward()` and `opt.step()`. It rescales the gradients in place and returns the norm measured before clipping, which is worth logging. Most LLM training runs clip at $c = 1.0$, and module 06 does the same.

The next cell checks the mechanics on one gradient, then trains the Kaiming network with plain SGD at a learning rate of 0.5, which is too high, once without clipping and once clipping at 1.0. Predict what each loss curve does.

In [ ]:
# The mechanics on one gradient: length capped, direction unchanged.
set_seed(0)
model = init_weights(deep_mlp(nn.ReLU), "kaiming")
F.cross_entropy(model(X_probe), y_probe).backward()
g_before = torch.cat([p.grad.flatten() for p in model.parameters()])
returned = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
g_after = torch.cat([p.grad.flatten() for p in model.parameters()])
print(f"norm before {g_before.norm().item():.2f} (returned {returned.item():.2f}) | norm after "
      f"{g_after.norm().item():.2f} | cosine similarity {F.cosine_similarity(g_before, g_after, dim=0).item():.4f}")


def train_deep_sgd(lr, max_norm, steps=300):
    set_seed(0)
    model = init_weights(deep_mlp(nn.ReLU), "kaiming")
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(0)
    losses, norms = [], []
    for step in range(steps):
        idx = torch.randint(0, len(X_train), (128,), generator=g)
        loss = F.cross_entropy(model(X_train[idx]), y_train[idx])
        opt.zero_grad()
        loss.backward()
        norms.append(torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm).item())
        opt.step()
        losses.append(loss.item())
    return np.array(losses), np.array(norms)


losses_raw, norms_raw = train_deep_sgd(0.5, max_norm=float("inf"))   # inf: measure the norm, never clip
losses_clip, norms_clip = train_deep_sgd(0.5, max_norm=1.0)

finite = lambda a: np.where(np.isfinite(a), a, np.nan)   # matplotlib skips nan; inf would break the log axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.semilogy(finite(losses_raw), label="no clipping")
ax1.semilogy(finite(losses_clip), label="clip at 1.0")
ax1.set_title("training loss"); ax1.set_xlabel("step"); ax1.legend()
ax2.semilogy(finite(norms_raw), label="no clipping")
ax2.semilogy(finite(norms_clip), label="clip at 1.0 (norm before clipping)")
ax2.axhline(1.0, color="gray", linestyle=":")
ax2.set_title("global gradient norm"); ax2.set_xlabel("step"); ax2.legend()
plt.show()

first_bad = int(np.argmax(~np.isfinite(losses_raw))) if not np.isfinite(losses_raw).all() else None
print(f"no clipping: largest finite loss {np.nanmax(finite(losses_raw)):.1e}, loss is nan or inf from step {first_bad}")
print(f"clip at 1.0: loss over the last 20 steps {losses_clip[-20:].mean():.3f}, "
      f"steps that were clipped {(norms_clip > 1.0).mean():.0%}")

What to notice:

- The single-gradient check: the norm drops to 1.00 and the cosine similarity between the gradient before and after is 1.0000, so only the length changed.
- Without clipping, the loss shoots up by many orders of magnitude and turns into `nan` within a few steps. The gradient norm explodes right along with it: the feedback loop described above.
- With clipping, the same learning rate trains normally.
- The clipped run's gradient norm (measured before clipping) is above 1.0 on every step, so here clipping is active all the time and acts like a cap on the step length. In a well-tuned run it's the other way around: the norm usually sits below the threshold and clipping only catches the occasional spike. Treat clipping as a safety net. It doesn't replace a sensible learning rate. That's why the `train` function in section 5 logs the norm on every run.

Clipping isn't the only defense. Very deep networks rely on three more fixes, all of which you'll meet:

- Normalization layers rescale activations inside the network so a bad scale can't compound from layer to layer: BatchNorm (exercise 4 at the end of this notebook) and LayerNorm (module 06).
- Residual connections, $x + f(x)$, give the gradient a direct path back through the additions (module 06).
- Adam's per-parameter scaling divides each gradient by a running estimate of its own size, so a layer whose gradients are 1,000 times too small still takes reasonably sized steps (module 02, section 11). It's another reason modern training is far less fragile than plain SGD, and part of why the shallow MLP and CNN in this notebook train fine with PyTorch's default init.

## 5. The training loop, with evaluation and logging

### Weights & Biases in two minutes

You logged in during setup (`uv run wandb login`). Every `wandb.init(...)` starts a run, every `wandb.log({...})` adds a point to its charts, and `wandb.finish()` closes it. Runs are grouped into projects. This notebook logs to a project called `ml-lab-fashion`, which you'll find at wandb.ai under your account.

The workflow that matters: change one thing, run, and compare against the previous run in the W&B dashboard. When you have 30 runs, the dashboard is how you remember what you tried and what worked. Tick runs on and off in the runs table and the charts overlay them.

If you're offline:

1. Set `WANDB_MODE=offline` before starting Jupyter, or run `os.environ["WANDB_MODE"] = "offline"` in a cell before the first training run. Runs are then saved locally in a `wandb/` folder inside this module, which git ignores.
2. When you're back online, upload them from a terminal:

```bash
cd ~/Developer/ML_Lab/lessons/05_training_and_wandb
uv run wandb sync wandb/offline-run-*
```

### The `train` function

This `train` function is what you'll reuse for the rest of the notebook. It adds five things to the loop from module 02:

1. A validation pass after every epoch, with the model in eval mode (dropout off) and gradients off.
2. A learning-rate schedule (used in section 8).
3. Accuracy tracking alongside loss.
4. Logging to W&B.
5. The global gradient norm from section 4, logged every 50 steps as `train/grad_norm`. An optional `grad_clip` config key turns on clipping at that threshold. It defaults to `None`, meaning no clipping: `train` then passes `float("inf")` as `max_norm`, so the norm is still measured but nothing is rescaled. Watch this chart in W&B. A norm that climbs steadily or spikes is an early warning of the blow-up you saw in section 4.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    model.train()
    return total_loss / n, correct / n


def lr_at(step, total_steps, base_lr, warmup_steps, schedule):
    if schedule == "constant":
        return base_lr
    if step < warmup_steps:                                  # linear warmup
        return base_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))  # cosine decay to 0


def train(model, config, train_loader, val_loader, run_name=None, run=None, augment=None):
    cfg = {"epochs": 5, "lr": 1e-3, "weight_decay": 0.0, "schedule": "constant", "warmup_frac": 0.05,
           "grad_clip": None, **config}
    if run is None:
        run = wandb.init(project=PROJECT, name=run_name, config=cfg)
    else:
        run.config.update(cfg, allow_val_change=True)

    model.to(device).train()
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    total_steps = cfg["epochs"] * len(train_loader)
    warmup_steps = max(1, int(cfg["warmup_frac"] * total_steps))
    max_norm = float("inf") if cfg["grad_clip"] is None else cfg["grad_clip"]   # inf: measure the norm, never clip
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    step, start = 0, time.time()

    for epoch in range(cfg["epochs"]):
        loss_sum, correct, seen = 0.0, 0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            if augment is not None:
                X = augment(X)
            lr = lr_at(step, total_steps, cfg["lr"], warmup_steps, cfg["schedule"])
            for group in opt.param_groups:
                group["lr"] = lr

            logits = model(X)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad()
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)   # norm before clipping
            opt.step()

            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            seen += len(y)
            if step % 50 == 0:
                run.log({"train/loss_step": loss.item(), "train/grad_norm": grad_norm.item(), "lr": lr}, step=step)
            step += 1

        val_loss, val_acc = evaluate(model, val_loader)
        history["train_loss"].append(loss_sum / seen)
        history["train_acc"].append(correct / seen)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        run.log({"epoch": epoch + 1, "train/loss": loss_sum / seen, "train/acc": correct / seen,
                 "val/loss": val_loss, "val/acc": val_acc}, step=step)
        if cfg["epochs"] <= 10 or (epoch + 1) % 10 == 0:
            print(f"epoch {epoch + 1:3}  train loss {loss_sum / seen:.4f} acc {correct / seen:.3f} | "
                  f"val loss {val_loss:.4f} acc {val_acc:.3f}  ({time.time() - start:.0f}s)")

    run.summary["best_val_acc"] = max(history["val_acc"])
    run.finish()
    return history

Train the MLP. Watch the printed W&B link and open it.

In [ ]:
set_seed(0)
mlp = MLP(hidden=256)
print(f"MLP parameters: {sum(p.numel() for p in mlp.parameters()):,}")
hist_mlp = train(mlp, {"epochs": 5, "lr": 1e-3, "model": "mlp"}, train_loader, val_loader, run_name="mlp-baseline")

Expect around 88% validation accuracy. One thing to notice: the reported train accuracy is measured while dropout is active and while the weights are still changing during the epoch, so it isn't exactly comparable to the validation number.

## 6. A convolutional network

The MLP flattens each image into 784 unrelated numbers, so it has to learn separately that a sleeve on the left and a sleeve on the right are the same thing. A convolution slides one small filter (here 3x3) across the whole image, reusing the same weights everywhere. That builds in two facts about images: nearby pixels are related, and a pattern means the same thing wherever it appears. Pooling then shrinks the image so later layers see larger regions.

In [ ]:
class CNN(nn.Module):
    def __init__(self, dropout=0.25, batchnorm=False):
        super().__init__()

        def block(c_in, c_out):
            layers = [nn.Conv2d(c_in, c_out, kernel_size=3, padding=1)]
            if batchnorm:
                layers.append(nn.BatchNorm2d(c_out))
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            return layers

        self.features = nn.Sequential(*block(1, 32), *block(32, 64))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)   # (N, 28, 28) -> (N, 1, 28, 28): conv layers expect a channel dimension
        return self.head(self.features(x))


# Trace the shapes through every layer. Predict each one first.
x = torch.zeros(1, 1, 28, 28)
cnn = CNN()
for layer in list(cnn.features) + list(cnn.head):
    x = layer(x)
    print(f"{layer.__class__.__name__:10} -> {tuple(x.shape)}")
print(f"CNN parameters: {sum(p.numel() for p in cnn.parameters()):,}")

Notice where the parameters live. The two conv layers have under 20,000 weights between them, and the first linear layer has about 400,000. Convolutions are cheap in parameters but expensive in compute, because each filter runs at every position.

In [ ]:
set_seed(0)
cnn = CNN()
hist_cnn = train(cnn, {"epochs": 5, "lr": 1e-3, "model": "cnn"}, train_loader, val_loader, run_name="cnn-baseline")

In [ ]:
plt.plot(range(1, 6), hist_mlp["val_acc"], "o-", label="MLP")
plt.plot(range(1, 6), hist_cnn["val_acc"], "o-", label="CNN")
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.title("MLP vs CNN")
plt.show()

Expect the CNN to land around 91%. In W&B, go to the project page and look at `val/acc` with both runs selected.

## 7. Overfitting

Give a big model very little data and it memorizes the training set instead of learning patterns that generalize. Here: a wide MLP, only 1,000 training images, 60 epochs, no regularization.

In [ ]:
small_idx = torch.randperm(len(X_train), generator=torch.Generator().manual_seed(1))[:1000]
small_loader = make_loader(X_train[small_idx], y_train[small_idx], 64, shuffle=True)

set_seed(0)
hist_overfit = train(MLP(hidden=512, dropout=0.0),
                     {"epochs": 60, "lr": 1e-3, "model": "mlp-512", "train_size": 1000},
                     small_loader, val_loader, run_name="small-data-no-reg")

Now the same model with the two standard fixes:

- Dropout randomly zeroes half the activations during training, so the network can't rely on any single path through it.
- Weight decay (in AdamW) shrinks every weight a little each step, which favors simpler functions. AdamW applies the decay directly to the weights, $w \leftarrow w - \eta \lambda w$ with learning rate $\eta$ and decay strength $\lambda$, separately from Adam's per-parameter gradient scaling. So every weight shrinks by the same fraction each step, whatever its gradient history (module 02, section 11).

In [ ]:
set_seed(0)
hist_reg = train(MLP(hidden=512, dropout=0.5),
                 {"epochs": 60, "lr": 1e-3, "weight_decay": 0.1, "dropout": 0.5, "model": "mlp-512", "train_size": 1000},
                 small_loader, val_loader, run_name="small-data-dropout-wd")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for hist, name in [(hist_overfit, "no regularization"), (hist_reg, "dropout + weight decay")]:
    ax1.plot(hist["train_acc"], label=f"train, {name}")
    ax1.plot(hist["val_acc"], "--", label=f"val, {name}")
    ax2.plot(hist["val_loss"], label=name)
ax1.set_title("accuracy"); ax1.set_xlabel("epoch"); ax1.legend(fontsize=8)
ax2.set_title("validation loss"); ax2.set_xlabel("epoch"); ax2.legend(fontsize=8)
plt.show()

What to read off these curves:

- Without regularization, training accuracy reaches 100% while validation accuracy stalls. The gap between them is overfitting.
- Validation loss falls, then climbs, even while validation accuracy holds steady. The model is getting more confident on the examples it gets wrong. In a real project, you'd stop training (or keep the checkpoint) where validation loss bottomed out. That's early stopping.
- With dropout and weight decay, the gap shrinks.

The best fix for overfitting is more data. With all 50,000 images, the same model barely overfits in 5 epochs.

## 8. Learning-rate schedules

A fixed learning rate is a compromise: big enough to make progress early, small enough to settle at the end. Schedules do both. Warmup ramps the rate up over the first few percent of steps (large updates at the very start, when the weights are random, can knock training off course), then cosine decay lowers it smoothly to zero. Almost every LLM is trained with warmup plus cosine or something close to it.

In [ ]:
total = 5 * len(train_loader)
lrs = [lr_at(s, total, 3e-3, int(0.05 * total), "cosine") for s in range(total)]
plt.plot(lrs); plt.xlabel("step"); plt.ylabel("learning rate"); plt.title("Warmup + cosine decay")
plt.show()

In [ ]:
set_seed(0)
cnn_cos = CNN()
hist_cos = train(cnn_cos, {"epochs": 5, "lr": 3e-3, "schedule": "cosine", "model": "cnn"},
                 train_loader, val_loader, run_name="cnn-cosine")

Compare `cnn-cosine` against `cnn-baseline` in W&B, and look at the `lr` chart for both runs. The cosine run uses a higher peak rate and usually finishes a bit better, because the low rate at the end lets it settle into a minimum.

## 9. Checkpoints, and the one-time test

A checkpoint saves the weights (`state_dict`) plus whatever you need to rebuild the model later. Checkpoints go in `checkpoints/`, which git ignores.

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
path = "checkpoints/cnn_cosine.pt"
torch.save({
    "model_state": cnn_cos.state_dict(),
    "model_args": {"dropout": 0.25, "batchnorm": False},
    "classes": CLASSES,
    "normalization": {"mean": mean.item(), "std": std.item()},
}, path)
print(f"saved {os.path.getsize(path) / 1e6:.1f} MB")

ckpt = torch.load(path, map_location=device)
restored = CNN(**ckpt["model_args"]).to(device)
restored.load_state_dict(ckpt["model_state"])
print("original val:", evaluate(cnn_cos, val_loader))
print("restored val:", evaluate(restored, val_loader))

Same numbers, so the checkpoint is complete. Note that the normalization stats are saved too: any new image has to be normalized exactly as the training data was, or predictions quietly degrade.

You've made your choices using validation data. Now, once, check the test set.

In [ ]:
test_loss, test_acc = evaluate(restored, test_loader)
print(f"test accuracy: {test_acc:.3f}")

restored.eval()
with torch.no_grad():
    preds = restored(X_test[:16].to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, p, t in zip(axes.flat, X_test[:16], preds, y_test[:16]):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{CLASSES[p]}\n(true: {CLASSES[t]})", fontsize=7, color="green" if p == t else "red")
    ax.axis("off")
plt.show()

## 10. A hyperparameter sweep (optional, about 10 minutes)

A sweep runs many trainings with different settings and lets W&B chart which settings worked. Random search over a sensible range is usually a better use of a fixed budget than a grid. Set `RUN_SWEEP = True` to run 8 short trainings, then open the sweep page W&B prints and look at the parallel coordinates chart.

In [ ]:
RUN_SWEEP = False

sweep_config = {
    "method": "random",
    "metric": {"name": "val/acc", "goal": "maximize"},
    "parameters": {
        "lr": {"distribution": "log_uniform_values", "min": 3e-4, "max": 1e-2},
        "dropout": {"values": [0.0, 0.25, 0.5]},
        "batch_size": {"values": [64, 128, 256]},
    },
}

def sweep_run():
    run = wandb.init()
    cfg = dict(run.config)
    set_seed(0)
    loader = make_loader(X_train, y_train, cfg["batch_size"], shuffle=True)
    model = CNN(dropout=cfg["dropout"])
    train(model, {"epochs": 2, "lr": cfg["lr"], "schedule": "cosine", "model": "cnn"}, loader, val_loader, run=run)

if RUN_SWEEP:
    sweep_id = wandb.sweep(sweep_config, project=PROJECT)
    wandb.agent(sweep_id, function=sweep_run, count=8)

## Wrap-up

Every model you'll train later, including the LoRA fine-tune in module 07, uses this loop and these diagnostics. When a run goes wrong, the checks from this notebook (the step-0 loss, per-layer activations and gradient norms, overfitting one batch, the gap between training and validation, the learning rate) are how you find out why. Experiment tracking is also how ML teams work: once you have ML engineers, "show me the W&B report" is how decisions get made.

You're done when you can look at a pair of training and validation curves and say whether the model is underfitting, overfitting, or has a learning-rate problem.

Going further:

- Karpathy's "Building makemore Part 3: Activations & Gradients, BatchNorm" (in his Neural Networks: Zero to Hero series) works through the same diagnostics as section 4 on a character-level model.
- The two initialization papers are short and readable: Glorot and Bengio, "Understanding the difficulty of training deep feedforward neural networks" (2010), and He et al., "Delving Deep into Rectifiers" (2015).

---
# Exercises

**Exercise 1: Confusion matrix.** For `cnn_cos` on the validation set, build the 10x10 confusion matrix (row = true class, column = predicted) without loops. Hint: `torch.bincount(true * 10 + pred, minlength=100)`. Plot it with `plt.imshow` and print the five most common mistakes. Which classes does the model confuse, and does that make sense to you as a human looking at the images?

**Exercise 2: Overfit one batch.** Take a single batch of 32 training images and train a fresh `CNN(dropout=0.0)` on only that batch for 200 steps. The loss should go to nearly zero. This is the first debugging check for any new model or training loop: if a model can't memorize 32 examples, there's a bug (wrong labels, a missing `zero_grad`, a broken loss), and no amount of tuning will fix it.

**Exercise 3: Learning-rate range test.** Start a fresh CNN at learning rate 1e-6 and multiply the rate by a constant factor every step so it reaches 1.0 after 200 steps. Record the loss at every step, stop early if the loss exceeds 4 times its minimum, and plot loss against learning rate on a log x-axis. A good learning rate is usually about 10 times smaller than the one where loss is lowest.

**Exercise 4: BatchNorm.** Train `CNN(batchnorm=True)` with the same settings as `cnn-cosine` and compare in W&B. BatchNorm normalizes each channel's activations within a batch, which usually makes training faster and less sensitive to the learning rate. Did it here?

**Exercise 5: Data augmentation.** On the 1,000-image `small_loader`, train a `CNN` for 40 epochs twice: once plain, and once with an `augment` function passed to `train` that randomly flips half of each batch left to right and shifts the whole batch by up to 2 pixels (`torch.roll`). Augmentation creates new training examples for free. Does validation accuracy improve?

---
# Solutions

In [ ]:
# Exercise 1
cnn_cos.eval()
preds, trues = [], []
with torch.no_grad():
    for X, y in val_loader:
        preds.append(cnn_cos(X.to(device)).argmax(dim=1).cpu())
        trues.append(y)
preds, trues = torch.cat(preds), torch.cat(trues)

cm = torch.bincount(trues * 10 + preds, minlength=100).reshape(10, 10)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(10), CLASSES, rotation=60, ha="right", fontsize=8)
plt.yticks(range(10), CLASSES, fontsize=8)
plt.xlabel("predicted"); plt.ylabel("true"); plt.colorbar()
plt.show()

mistakes = cm.clone()
mistakes.fill_diagonal_(0)
values, flat_idx = mistakes.flatten().topk(5)
for v, i in zip(values, flat_idx):
    print(f"{v.item():4} times: true {CLASSES[i // 10]!r} predicted as {CLASSES[i % 10]!r}")

In [ ]:
# Exercise 2
set_seed(0)
m = CNN(dropout=0.0).to(device)
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
xb, yb = next(iter(train_loader))
xb, yb = xb[:32].to(device), yb[:32].to(device)

for step in range(200):
    loss = F.cross_entropy(m(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 40 == 0 or step == 199:
        print(f"step {step:3}  loss {loss.item():.5f}")

In [ ]:
# Exercise 3
set_seed(0)
m = CNN().to(device)
opt = torch.optim.AdamW(m.parameters(), lr=1e-6)
lr, factor = 1e-6, (1.0 / 1e-6) ** (1 / 200)
lrs, losses = [], []
batches = iter(train_loader)

for step in range(200):
    try:
        X, y = next(batches)
    except StopIteration:
        batches = iter(train_loader)
        X, y = next(batches)
    for group in opt.param_groups:
        group["lr"] = lr
    loss = F.cross_entropy(m(X.to(device)), y.to(device))
    opt.zero_grad(); loss.backward(); opt.step()
    lrs.append(lr); losses.append(loss.item())
    if loss.item() > 4 * min(losses):
        break
    lr *= factor

best = lrs[int(np.argmin(losses))]
plt.semilogx(lrs, losses)
plt.xlabel("learning rate"); plt.ylabel("loss"); plt.title(f"LR range test (lowest loss at {best:.1e})")
plt.show()
print(f"suggested learning rate: about {best / 10:.1e}")

In [ ]:
# Exercise 4
set_seed(0)
hist_bn = train(CNN(batchnorm=True), {"epochs": 5, "lr": 3e-3, "schedule": "cosine", "model": "cnn-bn"},
                train_loader, val_loader, run_name="cnn-cosine-batchnorm")
print(f"best val acc  cosine: {max(hist_cos['val_acc']):.3f}   cosine + batchnorm: {max(hist_bn['val_acc']):.3f}")

In [ ]:
# Exercise 5
def augment(X):
    flip = torch.rand(len(X), device=X.device) < 0.5
    X = torch.where(flip[:, None, None], X.flip(-1), X)          # mirror half the batch left-right
    dx, dy = np.random.randint(-2, 3, size=2)
    return torch.roll(X, shifts=(int(dy), int(dx)), dims=(-2, -1))   # shift the batch a little

set_seed(0)
hist_plain = train(CNN(), {"epochs": 40, "lr": 1e-3, "model": "cnn", "train_size": 1000},
                   small_loader, val_loader, run_name="small-data-cnn")
set_seed(0)
hist_aug = train(CNN(), {"epochs": 40, "lr": 1e-3, "model": "cnn", "train_size": 1000, "augment": True},
                 small_loader, val_loader, run_name="small-data-cnn-augmented", augment=augment)
print(f"best val acc  plain: {max(hist_plain['val_acc']):.3f}   augmented: {max(hist_aug['val_acc']):.3f}")